In [1]:
from diffsky.diffndhist import tw_ndhist
from jax import random as jran

In [3]:
ndim = 2
MU_TARGET = np.ones(ndim)
sig = 0.1
COV = np.eye(ndim)*sig*sig

nbins = 10
sbins = np.linspace(0, 2, nbins+1)
NDBINS_LO = np.repeat(sbins[:-1], ndim).reshape((nbins, ndim))
NDBINS_HI = np.repeat(sbins[1:], ndim).reshape((nbins, ndim))


NPTS_PRED = 20_000

@jjit
def _predict_histogram(params, key, ndbins_lo, ndbins_hi):
    nddata = jran.multivariate_normal(key, mean=params, cov=COV, shape=(NPTS_PRED, ))
    ndsig = jnp.zeros_like(nddata) + sig
    return tw_ndhist(nddata, ndsig, ndbins_lo, ndbins_hi)


In [4]:
@jjit
def _mse(pred, target):
    diff = pred - target
    return jnp.mean(diff*diff)

@jjit
def _loss(params, data):
    key, ndbins_lo, ndbins_hi, target_hist = data
    pred_hist = _predict_histogram(params, key, ndbins_lo, ndbins_hi)
    return _mse(pred_hist, target_hist)

In [5]:
ran_key = jran.PRNGKey(0)
target_key, pred_key = jran.split(ran_key, 2)
TARGET_HIST = _predict_histogram(MU_TARGET, target_key, NDBINS_LO, NDBINS_HI)

In [6]:
p_init = MU_TARGET + sig/10

loss_data = pred_key, NDBINS_LO, NDBINS_HI, TARGET_HIST
_loss(p_init, loss_data)

Array(13261.523, dtype=float32)

In [7]:
from jax import grad

loss_grad = jjit(grad(_loss, argnums=0))
loss_grad(p_init, loss_data)

Array([1463550.1, 1460206.9], dtype=float32)